In [9]:
import cv2
import face_recognition
import numpy as np
import pandas as pd
from datetime import datetime
import os

IMAGE_PATH = "images"
TOLERANCE = 0.55
FRAME_SKIP = 3
WIDTH = 640
HEIGHT = 480

images = []
classNames = []

if not os.path.exists(IMAGE_PATH):
    print("Images folder not found")
    exit()

for file in os.listdir(IMAGE_PATH):

    path = os.path.join(IMAGE_PATH, file)

    img = cv2.imread(path)

    if img is None:
        continue

    images.append(img)

    name = os.path.splitext(file)[0]

    if "_" in name:
        name = name.split("_")[0]

    classNames.append(name)

print("Loaded:", classNames)

def findEncodings(images):

    encodeList = []

    for img in images:

        rgb = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB
        )

        encodes = face_recognition.face_encodings(rgb)

        if len(encodes) > 0:
            encodeList.append(encodes[0])

    return encodeList

print("Encoding Faces...")

encodeListKnown = findEncodings(images)

print("Encoding Complete")

def markAttendance(name):

    file = "attendance.csv"

    if not os.path.exists(file):

        df = pd.DataFrame(
            columns=["Name", "Date", "Time"]
        )

        df.to_csv(file, index=False)

    df = pd.read_csv(file)

    today = datetime.now().strftime("%Y-%m-%d")

    already_marked = (
        (df["Name"] == name) &
        (df["Date"] == today)
    ).any()

    if already_marked:
        return

    now = datetime.now()

    new_row = pd.DataFrame(
        [[
            name,
            today,
            now.strftime("%H:%M:%S")
        ]],
        columns=["Name", "Date", "Time"]
    )

    df = pd.concat(
        [df, new_row],
        ignore_index=True
    )

    df.to_csv(file, index=False)

    print(f"{name} Attendance Marked")

cap = cv2.VideoCapture(0)

cap.set(3, WIDTH)
cap.set(4, HEIGHT)

if not cap.isOpened():

    print("Camera not found")
    exit()

print("System Started")

frame_count = 0

face_locations = []
face_names = []

while True:

    success, img = cap.read()

    if not success:
        break

    img = cv2.flip(img, 1)

    frame_count += 1

    if frame_count % FRAME_SKIP == 0:

        small_img = cv2.resize(
            img,
            (0, 0),
            fx=0.25,
            fy=0.25
        )

        rgb_small = cv2.cvtColor(
            small_img,
            cv2.COLOR_BGR2RGB
        )

        faces = face_recognition.face_locations(
            rgb_small,
            model="hog"
        )

        encodes = face_recognition.face_encodings(
            rgb_small,
            faces
        )

        face_locations = faces
        face_names = []

        for encodeFace in encodes:

            name = "UNKNOWN"

            matches = face_recognition.compare_faces(
                encodeListKnown,
                encodeFace,
                tolerance=TOLERANCE
            )

            faceDis = face_recognition.face_distance(
                encodeListKnown,
                encodeFace
            )

            if len(faceDis) > 0:

                matchIndex = np.argmin(faceDis)

                if matches[matchIndex]:

                    name = classNames[
                        matchIndex
                    ].upper()

                    markAttendance(name)

            face_names.append(name)

    for (faceLoc, name) in zip(
        face_locations,
        face_names
    ):

        y1, x2, y2, x1 = faceLoc

        y1 *= 4
        x2 *= 4
        y2 *= 4
        x1 *= 4

        color = (
            (0, 255, 0)
            if name != "UNKNOWN"
            else (0, 0, 255)
        )

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            color,
            2
        )

        cv2.rectangle(
            img,
            (x1, y2 - 35),
            (x2, y2),
            color,
            cv2.FILLED
        )

        cv2.putText(
            img,
            name,
            (x1 + 6, y2 - 6),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2
        )

    cv2.imshow(
        "Face Attendance System",
        img
    )

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()

cv2.destroyAllWindows()

print("System Closed")

Loaded: ['amardeep', 'anil', 'gaurav (2)', 'istkar', 'lucky priya', 'mritunjay', 'palak', 'Param jeet sir', 'Rajat', 'Vipul', 'Yogesh']
Encoding Faces...
Encoding Complete
System Started
VIPUL Attendance Marked
System Closed
